<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Autonomous Traders</h2>
            <span style="color:#ff7800;">MCP サーバーのツールとリソースによって動かされる自律型エージェントを示すための、株式取引シミュレーションです。
            </span>
        </td>
    </tr>
</table>

### Week 6 Day 4

Capstone プロジェクトへようこそ!

# Autonomous Traders

4人のトレーダーと1人のリサーチャーによる株式取引シミュレーションで、MCP サーバーのチームとそのツールやリソースによって動いています。

1. 私たちのエンジニアリングチームが書いた、自家製の Accounts サーバー
2. 何かが起きたときに知らせてくれるプッシュ通知
3. ライブまたはシミュレートされた株価のための市場データ
4. ローカルのヘッドレスブラウザ経由で Web ページを読むための Fetch
5. Web 検索のための Tavily
6. リサーチャーが書き込み、読み返す知識グラフである Memory

さらに、各トレーダーのアカウントと投資戦略を読むためのリソースもあります。

このシステム全体は、すでに `backend` パッケージの中にあります。今日は、まず主要な部品を一通り見て回り、1人のトレーダーを組み立てて実行してその動きを観察し、その後ダッシュボードを立ち上げて、チーム全体が取引する様子を見ます。

## アーキテクチャ

各部品がどう組み合わさっているか見てみましょう。4人のトレーダーはそれぞれ、accounts、プッシュ通知、市場データ用の独自の MCP サーバーを持つエージェントで、リサーチャーエージェントをツールとして呼び出します。リサーチャー自身もエージェントで、ページの取得、Web 検索、記憶のための独自の MCP サーバーを持っています。オレンジがエージェント、青が MCP サーバーで、全体で6つの MCP サーバーがあります。

エージェントに「Trader」や「Researcher」のような名前を付けましたが、このアーキテクチャの狙いは、適切なプロンプトとツールによってコンテキストをうまく管理し、信頼できる結果を得ることでした。人間のチームがそう構成されているからというだけの理由でエージェントの責務を割り振ってしまう、という落とし穴を避けることが重要です。

<img src="../assets/architecture.png" width="820" />

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">もう一度言います --</h2>
            <span style="color:#ff7800;">これを実際の取引の判断には使わないでください!!
            </span>
        </td>
    </tr>
</table>

In [ ]:
from dotenv import load_dotenv
import json
from contextlib import AsyncExitStack
from agents import Runner, trace, add_trace_processor
from IPython.display import Markdown, display
from backend.market import get_share_price
from backend.accounts import Account
from backend.accounts_client import read_accounts_resource
from backend.reset import reset_traders
from backend.mcp_servers import trader_mcp_servers, researcher_mcp_servers
from backend.traders import get_researcher, get_researcher_tool, Trader
from backend.tracers import LogTracer

load_dotenv(override=True)

### Windows での注意点

ノートブックからローカルの MCP サーバーを起動すると、Windows では厄介な問題にぶつかります。サーバーは起動時の出力を stderr に書き込みますが、Windows の Jupyter カーネルの中ではこのストリームの裏に実際のファイルハンドルがないため、起動が `io.UnsupportedOperation: fileno` エラーで失敗します。一方、Mac と Linux では影響がありません。

対処法は、サーバーの stderr をヌルデバイスに送ることです。これにより、サーバーは常に書き込める実在の場所を持つことになります。次のセルでこれを一度だけ行えば、以降のすべての MCP サーバーが問題なく起動できるようになります。Mac と Linux では、サーバーの起動時バナーがノートブックに表示されなくなるだけで、他に影響はありません。

In [ ]:
# Windows では、Jupyter カーネルから起動した stdio MCP サーバーが、実際のファイルディスクリプタを
# 持たない stderr ストリームに書き込もうとして io.UnsupportedOperation: fileno でクラッシュする。
# サーバーの stderr をヌルデバイスに送ることで、常に書き込める実在の場所を用意する。Mac と Linux では影響がない。
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## backend の見どころツアー

取引フロアは、すでに `backend` の中に Python パッケージとして組み立てられています。実行する前に、それが提供してくれる部品をいくつか見てみましょう。

### 市場データ

`get_share_price` は、ある銘柄の最新の価格を返します。Massive の API キーがあればライブの市場データを使い、なければシミュレーターにフォールバックするので、どちらの場合でもフロアは動作します。

In [ ]:
get_share_price("AAPL")

### トレーダーのアカウント

各トレーダーはアカウントを持っています。残高、保有株、取引履歴、そして投資戦略です。`Account` モデルは、私たちの自家製 Accounts サーバーの中心です。4人のトレーダーを初期戦略にリセットし、そのうち1人を見てみましょう。

#### Ed のメモ:

自分のトレーダーをリセットしたくないので、最初の行をコメントアウトしています!

In [ ]:
# reset_traders() 
warren = Account.get("Warren")
print("Balance:", warren.balance)
print("Strategy:", warren.get_strategy())

### MCP サーバーと、そのツール

トレーダーとリサーチャーは、それぞれ独自の MCP サーバー群を持っており、`mcp_servers.py` がそれらを組み立ててくれます。それぞれを起動し、公開されているツールの数を数えてみましょう。

In [ ]:
servers = trader_mcp_servers() + researcher_mcp_servers("Warren")
count = 0
for server in servers:
    async with server:
        tools = await server.list_tools()
        count += len(tools)
print(f"We have {len(servers)} MCP servers, and {count} tools")

## リサーチャー

トレーダー自身は Web 検索を行いません。代わりに、リサーチャーエージェントをツールの1つとして呼び出します。リサーチャーは独自の MCP サーバーを持っています。ページを読むための Fetch、Web 検索のための Tavily、そして書き込みと読み返しを行う Memory です。

Tavily は、単純な検索から重量級のディープリサーチモードまで、複数のツールを提供しています。ここではそのサーバーを `tavily_search` に制限し、リサーチャーを高速かつ集中させています。エージェントに見せるツールを選ぶこと自体が、コンテキストエンジニアリングです。

エージェントをツールとしてラップすることは、ハンドオフとは異なります。ツールを使う場合、トレーダーは制御を保持したまま、リサーチャーの答えを受け取ります。一方、ハンドオフでは会話全体が渡されてしまいます。

In [ ]:
async with AsyncExitStack() as stack:
    servers = [await stack.enter_async_context(server) for server in researcher_mcp_servers("Warren")]
    researcher = await get_researcher(servers, "gpt-5.4-mini")
    with trace("Researcher"):
        result = await Runner.run(researcher, "What's the latest news on Amazon?", max_turns=30)
display(Markdown(result.final_output))

### トレースを見てみましょう

https://platform.openai.com/traces

### リサーチャーをツールとしてラップする

トレーダーはリサーチャーと直接話すわけではありません。リサーチャーエージェント全体をツールに変換し、トレーダーはそれを他のツールと同じように呼び出します。`get_researcher_tool` はこれを `researcher.as_tool(...)` で実現しています。

In [ ]:
researcher_tool = await get_researcher_tool(researcher_mcp_servers("Warren"), "gpt-5.4-mini")
print("Tool name:", researcher_tool.name)
print("Description:", researcher_tool.description)

## トレーダー

`traders.py` の `Trader` クラスが、これらすべてをまとめます。リサーチャーをツールとして組み立て、そのツールとともに、トレーダーの MCP サーバー(accounts、push、市場データ)の上にトレーダーエージェントを作成し、トレーダーの戦略と現在のアカウントから組み立てたメッセージに対してそれを実行します。

もう1つ見ておく価値のある部品があります。OpenAI Agents SDK ではそのトレーシング機構に接続できるので、各エージェントがコード上で何をしているかを追跡できます。`tracers.py` には、各トレーダーのステップをデータベースに記録するカスタムトレースプロセッサがあり、これによって彼らの思考をダッシュボード上に表示できるようになっています。今それを登録し、それから Warren を実行します。

In [ ]:
add_trace_processor(LogTracer())
warren = Trader("Warren", "Patience", "gpt-5.4-mini")
await warren.run()

### Warren の結果は?

MCP リソースを通じてアカウントを読み返すと、Warren が行った取引とポートフォリオの状態がわかります。

In [ ]:
resources = await read_accounts_resource("Warren")
info = json.loads(resources)
print(info["transactions"][-1])

## チーム全体を、ループで

`trading_floor.py` はバックエンドの完全な実装です。4人のトレーダー全員を作成し、タイマーで実行します。

```
while True:
    await asyncio.gather(*[trader.run() for trader in traders])
    await asyncio.sleep(RUN_EVERY_N_MINUTES * 60)
```

`.env` ファイルの中のいくつかの任意設定でこれを制御できます。

`RUN_EVERY_N_MINUTES=60` は、チームがどのくらいの頻度で実行されるかを設定します。デフォルトは60分ごとです。

`RUN_EVEN_WHEN_MARKET_IS_CLOSED=False` は、トレーダーが市場時間外にも実行されるかどうかを決めます。

`USE_MANY_MODELS=False` は、デフォルトで4人のトレーダー全員を gpt-5.4-mini 上で実行します。これを true に設定すると、各トレーダーに異なるモデルが割り当てられます。GPT-5.5、DeepSeek V4、Gemini 3.5 Flash、Grok 4.3 です。

## ダッシュボード

では、実際に見てみましょう。ダッシュボードは `demo` パッケージの中にあり、`app.py` がそれを起動します。

新しいターミナルを開き(ターミナルパネルの「+」)、このディレクトリに移動して、ダッシュボードを実行します。

`cd 6_mcp`

`uv run app.py`

次に、チームに取引をさせるために、別のターミナルを開き、同じディレクトリに移動して、エンジンを起動します。

`cd 6_mcp`

`uv run -m backend.trading_floor`

ダッシュボードを見て、あなたの取引チームが実際に動く様子を観察してください。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">API の使用量に注意してください</h2>
            <span style="color:#ff7800;">このエンジンは、1時間ごと、あるいは設定した間隔でループを続けます。API の使用量に注意し、十分見終わったら止めてください。私はこれを何時間も楽しく眺めてしまいがちですが、あなたもそうなることを願っています。
            </span>
        </td>
    </tr>
</table>

## もう少しです

自律的な取引フロアが手に入りました。4人のトレーダー、1人のリサーチャー、6つの MCP サーバー、永続的な記憶、そしてライブのダッシュボードです。

最後のラボでは、これに本番用のフロントエンド、同じバックエンドと通信する別の Web アプリを与えます。